# FedSSL — Exploration & Visualization Notebook

This notebook provides:
1. **EDA** — Sample X-ray visualization from NIH, Shenzhen, Montgomery
2. **MAE Visualization** — Masking and reconstruction on a sample image
3. **t-SNE** — Encoder embeddings (TB vs Normal) before and after SSL
4. **Training Curves** — Per-round federated loss
5. **ROC Curve** — Final evaluation on Montgomery test set

---
> **Note**: Run `python src/federated/simulation.py --config configs/default.yaml` first to generate checkpoints and logs before executing cells 3–5.

In [ ]:
# ── Setup: add repo root to path ─────────────────────────────────────────────
import sys, os
from pathlib import Path

REPO_ROOT = Path(os.getcwd()).parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image

sns.set_theme(style='dark', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

print('Setup complete. Repo root:', REPO_ROOT)

---
## 1. EDA — Sample X-Ray Visualization

In [ ]:
from src.datasets.loader import NIHDataset, ShenzhenDataset, MontgomeryDataset, get_eval_transform

# ── Config paths (adjust if needed) ──────────────────────────────────────────
NIH_PATH        = str(REPO_ROOT / 'data/raw/NIH')
SHENZHEN_PATH   = str(REPO_ROOT / 'data/raw/Shenzhen')
MONTGOMERY_PATH = str(REPO_ROOT / 'data/raw/Montgomery')

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def denormalize(tensor, mean=IMAGENET_MEAN, std=IMAGENET_STD):
    """Reverse ImageNet normalization for visualization."""
    t = tensor.clone().cpu().permute(1, 2, 0).numpy()
    t = t * np.array(std) + np.array(mean)
    return np.clip(t, 0, 1)

def show_dataset_samples(dataset, title, n=4, label_map=None):
    fig, axes = plt.subplots(1, n, figsize=(3.5*n, 3.5))
    fig.suptitle(title, fontsize=14, fontweight='bold', y=1.02)
    for i in range(min(n, len(dataset))):
        item = dataset[i]
        if isinstance(item, tuple):
            img_tensor, label = item[0], item[1]
            subtitle = f'Label: {label_map[label] if label_map else label}'
        else:
            img_tensor = item[0] if isinstance(item, (list, tuple)) else item
            subtitle = 'Unlabeled'
        img = denormalize(img_tensor)
        axes[i].imshow(img, cmap='gray')
        axes[i].set_title(subtitle, fontsize=10)
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()

label_map = {0: 'Normal', 1: 'TB'}
transform  = get_eval_transform(224)

# NIH — unlabeled
nih_ds = NIHDataset(NIH_PATH, transform=transform, two_view=False)
print(f'NIH dataset: {len(nih_ds)} images')
if len(nih_ds) > 0:
    show_dataset_samples(nih_ds, 'NIH ChestX-ray14 (Unlabeled — SSL Pretraining)')

# Shenzhen — TB / Normal
shz_ds = ShenzhenDataset(SHENZHEN_PATH, transform=transform)
print(f'Shenzhen dataset: {len(shz_ds)} images | TB: {sum(shz_ds.get_labels())} | Normal: {len(shz_ds)-sum(shz_ds.get_labels())}')
if len(shz_ds) > 0:
    show_dataset_samples(shz_ds, 'Shenzhen TB Dataset (Few-Shot Fine-tuning)', label_map=label_map)

# Montgomery — held-out test
mont_ds = MontgomeryDataset(MONTGOMERY_PATH, transform=transform)
print(f'Montgomery dataset: {len(mont_ds)} images | TB: {sum(mont_ds.get_labels())} | Normal: {len(mont_ds)-sum(mont_ds.get_labels())}')
if len(mont_ds) > 0:
    show_dataset_samples(mont_ds, 'Montgomery TB Dataset (Held-Out Test Set)', label_map=label_map)

---
## 2. MAE Masking & Reconstruction Visualization

In [ ]:
from src.models.mae import build_mae
from src.utils.config import load_config
import torchvision.transforms as T

config = load_config(str(REPO_ROOT / 'configs/default.yaml'))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Build MAE with random init
mae = build_mae(
    backbone=config.model.backbone,
    embed_dim=config.model.embed_dim,
    mask_ratio=config.model.mask_ratio,
    decoder_depth=config.model.decoder_depth,
).to(device)
mae.eval()

# Load a real image, or use synthetic noise if no data available
def get_sample_image(dataset, idx=0):
    if len(dataset) > 0:
        item = dataset[idx]
        return item[0] if isinstance(item, tuple) else item
    else:
        print('[WARNING] No real images found — using synthetic noise.')
        return torch.randn(3, 224, 224)

img_tensor = get_sample_image(shz_ds if len(shz_ds) > 0 else nih_ds).unsqueeze(0).to(device)

with torch.no_grad():
    # Get patches
    patches = mae.patchify(img_tensor)   # (1, 196, 768)

    # Tokenize & apply masking only
    tokens = mae._tokenize(img_tensor)
    _, mask, ids_restore = mae.random_masking(tokens, mae.mask_ratio)

    # Create masked image for display
    mask_np = mask[0].cpu().numpy()   # (196,)
    h = w = 14  # 224/16
    mask_2d = mask_np.reshape(h, w)

    # Full forward pass to get reconstruction
    loss, pred, mask_out = mae(img_tensor)
    pred_img = mae.unpatchify(pred)[0].cpu()

# ── Plot ──────────────────────────────────────────────────────────────────────
original   = denormalize(img_tensor[0].cpu())
prediction = denormalize(pred_img)

# Build masked image (grey out masked patches)
masked_img = original.copy()
p = 16
for row in range(h):
    for col in range(w):
        if mask_2d[row, col] == 1:
            masked_img[row*p:(row+1)*p, col*p:(col+1)*p, :] = 0.5

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
titles = ['Original Image', f'Masked ({int(mae.mask_ratio*100)}% patches hidden)', 'MAE Reconstruction']
imgs   = [original, masked_img, prediction]
for ax, title, img in zip(axes, titles, imgs):
    ax.imshow(img, cmap='gray')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.axis('off')

plt.suptitle(f'MAE Masked Autoencoding | Loss: {loss.item():.4f}', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()
print(f'Reconstruction MSE Loss: {loss.item():.6f}')

---
## 3. t-SNE Embedding Visualization (Before & After SSL)

In [ ]:
from sklearn.manifold import TSNE
from src.models.encoder import get_encoder
from torch.utils.data import DataLoader
import warnings
warnings.filterwarnings('ignore')

def extract_embeddings_for_tsne(encoder, dataset, device, max_samples=200):
    encoder.eval()
    loader = DataLoader(dataset, batch_size=16, shuffle=True)
    embeddings, labels = [], []
    total = 0
    with torch.no_grad():
        for batch in loader:
            if total >= max_samples:
                break
            imgs, lbls = batch if isinstance(batch, (list, tuple)) and len(batch)==2 else (batch, torch.zeros(batch.shape[0]))
            imgs = imgs.to(device)
            emb = encoder(imgs).cpu().numpy()
            embeddings.append(emb)
            labels.extend(lbls.tolist() if hasattr(lbls, 'tolist') else lbls)
            total += imgs.shape[0]
    if not embeddings:
        return np.random.randn(50, 512), [i%2 for i in range(50)]
    return np.vstack(embeddings)[:max_samples], labels[:max_samples]

def plot_tsne(embeddings, labels, title, ax):
    tsne = TSNE(n_components=2, perplexity=min(30, len(embeddings)//4), random_state=42, max_iter=1000)
    z2d = tsne.fit_transform(embeddings)
    label_names = {0: 'Normal', 1: 'TB'}
    colors = {0: '#4A90D9', 1: '#E74C3C'}
    for cls in [0, 1]:
        idx = [i for i, l in enumerate(labels) if l == cls]
        if idx:
            ax.scatter(z2d[idx, 0], z2d[idx, 1], c=colors[cls],
                      label=label_names[cls], alpha=0.7, s=35, edgecolors='white', linewidths=0.4)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.set_xlabel('t-SNE dim 1'); ax.set_ylabel('t-SNE dim 2')
    ax.grid(True, alpha=0.2)

# ── Random-init encoder (before SSL) ─────────────────────────────────────────
encoder_random = get_encoder(config.model.backbone, config.model.embed_dim).to(device)

# ── SSL-trained encoder (after SSL): load best checkpoint ────────────────────
encoder_trained = get_encoder(config.model.backbone, config.model.embed_dim).to(device)
best_ckpt = REPO_ROOT / 'experiments/checkpoints/best_encoder.pt'
if best_ckpt.exists():
    ckpt = torch.load(str(best_ckpt), map_location=device)
    encoder_trained.load_state_dict(ckpt['encoder_state_dict'])
    print(f'Loaded best encoder (AUC={ckpt.get("best_auc", "N/A")})')
else:
    print('[WARNING] best_encoder.pt not found — showing random embeddings for both.')

# Choose dataset for visualization
viz_ds = shz_ds if len(shz_ds) > 0 else mont_ds
if len(viz_ds) == 0:
    print('[WARNING] No real labeled data — using synthetic embeddings for t-SNE.')
    embs_random  = np.random.randn(100, config.model.embed_dim)
    embs_trained = np.random.randn(100, config.model.embed_dim) + np.tile([3,-3]*(config.model.embed_dim//2), 100).reshape(100,-1)
    labels_viz   = [i%2 for i in range(100)]
else:
    embs_random,  labels_viz = extract_embeddings_for_tsne(encoder_random,  viz_ds, device)
    embs_trained, _          = extract_embeddings_for_tsne(encoder_trained, viz_ds, device)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))
plot_tsne(embs_random,  labels_viz, 'Before SSL (Random Init)', ax1)
plot_tsne(embs_trained, labels_viz, 'After FedSSL Pretraining', ax2)
plt.suptitle('t-SNE of Encoder Embeddings — TB vs Normal', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 4. Per-Round Federated Training Loss Curves

In [ ]:
import json

log_path = REPO_ROOT / 'experiments/logs/training_log.json'

if log_path.exists():
    with open(str(log_path)) as f:
        rounds_data = json.load(f)

    rounds      = [r['round'] + 1 for r in rounds_data]
    ssl_losses  = [r.get('mean_ssl_loss', float('nan')) for r in rounds_data]
    aucs        = [r['eval_metrics']['auc'] if 'eval_metrics' in r else None for r in rounds_data]
    auc_rounds  = [r+1 for r, a in zip([d['round'] for d in rounds_data], aucs) if a is not None]
    auc_vals    = [a for a in aucs if a is not None]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # SSL Loss
    ax1.plot(rounds, ssl_losses, color='#4A90D9', linewidth=2.2, marker='o', markersize=4)
    ax1.fill_between(rounds, ssl_losses, alpha=0.15, color='#4A90D9')
    ax1.set_xlabel('Federated Round', fontsize=12)
    ax1.set_ylabel('Mean SSL Loss (MSE)', fontsize=12)
    ax1.set_title('Federated SSL Training Loss', fontsize=13, fontweight='bold')
    ax1.grid(True, alpha=0.3)

    # AUC over rounds
    if auc_vals:
        ax2.plot(auc_rounds, auc_vals, color='#2ECC71', linewidth=2.2, marker='s', markersize=6)
        ax2.axhline(y=0.85, color='#E74C3C', linestyle='--', linewidth=1.5, label='Target AUC=0.85')
        ax2.fill_between(auc_rounds, auc_vals, alpha=0.15, color='#2ECC71')
        ax2.set_xlabel('Federated Round', fontsize=12)
        ax2.set_ylabel('Montgomery AUC', fontsize=12)
        ax2.set_title('Montgomery AUC vs Federated Round', fontsize=13, fontweight='bold')
        ax2.legend(fontsize=11)
        ax2.set_ylim(0.4, 1.0)
        ax2.grid(True, alpha=0.3)

    plt.suptitle('FedSSL Training Progress', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

else:
    print('[INFO] No training log found. Run the simulation first:')
    print('  python src/federated/simulation.py --config configs/default.yaml')
    # Show demo synthetic curves
    demo_rounds = list(range(1, 21))
    demo_losses = [0.5 * np.exp(-0.15*r) + 0.05 + 0.01*np.random.randn() for r in demo_rounds]
    demo_aucs_r = [5, 10, 15, 20]
    demo_aucs   = [0.62, 0.74, 0.81, 0.87]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(demo_rounds, demo_losses, color='#4A90D9', linewidth=2.2, marker='o', markersize=4, label='Synthetic Demo')
    ax1.set_title('[DEMO] Federated SSL Training Loss', fontsize=13, fontweight='bold')
    ax1.set_xlabel('Round'); ax1.set_ylabel('SSL Loss'); ax1.grid(True, alpha=0.3); ax1.legend()

    ax2.plot(demo_aucs_r, demo_aucs, color='#2ECC71', linewidth=2.2, marker='s', markersize=6, label='Synthetic Demo')
    ax2.axhline(y=0.85, color='#E74C3C', linestyle='--', linewidth=1.5, label='Target AUC=0.85')
    ax2.set_title('[DEMO] Montgomery AUC vs Round', fontsize=13, fontweight='bold')
    ax2.set_xlabel('Round'); ax2.set_ylabel('AUC'); ax2.set_ylim(0.4, 1.0); ax2.grid(True, alpha=0.3); ax2.legend()

    plt.tight_layout(); plt.show()

---
## 5. Final ROC Curve on Montgomery Test Set

In [ ]:
from sklearn.metrics import roc_curve, auc as sklearn_auc
from src.client.local_train import finetune_local, evaluate_on_montgomery, _extract_embeddings
from src.models.proto_head import PrototypicalHead
from torch.utils.data import DataLoader

best_ckpt = REPO_ROOT / 'experiments/checkpoints/best_encoder.pt'

if not best_ckpt.exists():
    print('[INFO] No checkpoint found. Showing demo ROC curve.')
    # Demo synthetic ROC
    np.random.seed(42)
    y_true  = np.array([i%2 for i in range(200)])
    y_score = np.clip(y_true * 0.6 + np.random.randn(200)*0.25, 0, 1)
    fpr, tpr, _ = roc_curve(y_true, y_score)
    roc_auc = sklearn_auc(fpr, tpr)

    fig, ax = plt.subplots(figsize=(7, 6))
    ax.plot(fpr, tpr, color='#4A90D9', lw=2.5, label=f'[DEMO] ROC (AUC = {roc_auc:.4f})')
    ax.plot([0,1],[0,1], color='gray', linestyle='--', lw=1.5, label='Random Classifier')
    ax.set_xlabel('False Positive Rate', fontsize=12)
    ax.set_ylabel('True Positive Rate', fontsize=12)
    ax.set_title('[DEMO] ROC Curve — Montgomery Test Set', fontsize=13, fontweight='bold')
    ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

else:
    from src.models.encoder import get_encoder
    encoder_eval = get_encoder(config.model.backbone, config.model.embed_dim).to(device)
    ckpt = torch.load(str(best_ckpt), map_location=device)
    encoder_eval.load_state_dict(ckpt['encoder_state_dict'])
    encoder_eval.eval()

    shz_loader  = DataLoader(shz_ds, batch_size=16, shuffle=False) if len(shz_ds) > 0 else None
    mont_loader = DataLoader(mont_ds, batch_size=16, shuffle=False) if len(mont_ds) > 0 else None

    if shz_loader and mont_loader:
        # Fine-tune on Shenzhen
        proto_head, _ = finetune_local(0, encoder_eval, shz_loader, config, device)

        # Collect Montgomery predictions
        from src.client.local_train import _sample_kshot
        support_emb, support_lbl = _extract_embeddings(encoder_eval, shz_loader, device)
        sup_idx, _ = _sample_kshot(support_lbl, k=config.finetuning.few_shot_k, num_classes=2)
        prototypes = proto_head.compute_prototypes(support_emb[sup_idx].to(device), support_lbl[sup_idx].to(device))

        all_probs, all_labels = [], []
        with torch.no_grad():
            for imgs, labels in mont_loader:
                emb = encoder_eval(imgs.to(device))
                _, probs = proto_head.predict(emb, prototypes)
                all_probs.append(probs[:,1].cpu())
                all_labels.append(labels)
        y_score = torch.cat(all_probs).numpy()
        y_true  = torch.cat(all_labels).numpy()

        fpr, tpr, _ = roc_curve(y_true, y_score)
        roc_auc = sklearn_auc(fpr, tpr)

        fig, ax = plt.subplots(figsize=(7, 6))
        ax.plot(fpr, tpr, color='#4A90D9', lw=2.5, label=f'FedSSL ROC (AUC = {roc_auc:.4f})')
        ax.fill_between(fpr, tpr, alpha=0.1, color='#4A90D9')
        ax.plot([0,1],[0,1], color='gray', linestyle='--', lw=1.5, label='Random Classifier')
        ax.axvline(x=0.2, color='#E74C3C', linestyle=':', alpha=0.5)
        ax.set_xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
        ax.set_ylabel('True Positive Rate (Sensitivity)', fontsize=12)
        ax.set_title('ROC Curve — FedSSL on Montgomery Test Set', fontsize=13, fontweight='bold')
        ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
        plt.tight_layout(); plt.show()
        print(f'Final Montgomery AUC: {roc_auc:.4f}')
    else:
        print('[WARNING] Real dataset not found — re-run with real data.')